# Building a training set from soundscapes

This notebook joins audio to labels. The labels table has one row per five second segment. For each row, the matching slice of audio is cut out, turned into a mel-spectrogram, and paired with the species present in that segment. The result is a stack of labelled spectrograms, which is the input a model needs.

In [7]:
import os
import json
import numpy as np
import pandas as pd
import librosa
from sklearn.preprocessing import MultiLabelBinarizer

SR = 32000            # sample rate everything is standardised to
DATA_DIR = '../data'  # where the audio and CSVs live

# the labels table: one row per 5-second segment
ss = pd.read_csv('../data/train_soundscapes_labels.csv')

## Selecting the files on disk

Only five soundscape files have been downloaded so far, so the labels table is filtered down to just those. That gives the segments that can actually be processed.

In [8]:
# the five files currently downloaded, one per site
have = [
    "BC2026_Train_0039_S22_20211231_201500.ogg",
    "BC2026_Train_0005_S08_20250607_070007.ogg",
    "BC2026_Train_0014_S13_20220228_214500.ogg",
    "BC2026_Train_0061_S19_20250418_210000.ogg",
    "BC2026_Train_0057_S15_20250617_060000.ogg",
]

# keep only the rows whose file is one we actually have
subset = ss[ss['filename'].isin(have)].copy()

print("segments to process:", len(subset))
print(subset.head())

segments to process: 120
                                    filename     start       end  \
0  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:00  00:00:05   
1  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:05  00:00:10   
2  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:10  00:00:15   
3  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:15  00:00:20   
4  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:20  00:00:25   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380  
3  22961;23158;24321;517063;65380  
4  22961;23158;24321;517063;65380  


## Cutting a single segment

The start and end times in the table are text such as `00:00:05`. They are converted to seconds, then `librosa.load` reads only that slice using its offset and duration arguments. This is tested on one row first, to confirm the slice is exactly five seconds (160000 samples at 32 kHz) before processing all of them.

In [9]:
def time_to_seconds(t):
    # "00:00:05" -> 5
    h, m, s = t.split(':')
    return int(h) * 3600 + int(m) * 60 + int(s)

# test on the second row: the 5s to 10s slice of the first file
row = subset.iloc[1]
start = time_to_seconds(row['start'])
end = time_to_seconds(row['end'])

path = os.path.join(DATA_DIR, row['filename'])

# offset = start reading here, duration = read this many seconds
y, sr = librosa.load(path, sr=SR, offset=start, duration=end - start)

print("start:", start, "end:", end)
print("samples loaded:", len(y), " expected:", SR * (end - start))

start: 5 end: 10
samples loaded: 160000  expected: 160000


## Processing all segments

The same cut and transform is applied to every segment. Slices that come up slightly short, which can happen on the final segment of a file, are padded to the full length so every spectrogram has the same shape. Each segment's labels are split from their semicolon separated string into a list.

In [10]:
def make_spectrogram(y):
    # waveform -> mel-spectrogram -> decibel scale (an image of the sound)
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128, fmin=40, fmax=15000)
    return librosa.power_to_db(mel, ref=np.max)

specs = []   # will hold one spectrogram per segment
labels = []  # will hold one label list per segment

for i, row in subset.iterrows():
    start = time_to_seconds(row['start'])
    end = time_to_seconds(row['end'])
    path = os.path.join(DATA_DIR, row['filename'])

    # load just this segment's audio
    y, sr = librosa.load(path, sr=SR, offset=start, duration=end - start)

    # pad short slices so every segment is the same length
    target = SR * (end - start)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))

    specs.append(make_spectrogram(y))
    labels.append(row['primary_label'].split(';'))  # "a;b;c" -> [a, b, c]

specs = np.array(specs)

print("spectrogram stack shape:", specs.shape)
print("number of label lists:", len(labels))
print("first segment labels:", labels[0])

spectrogram stack shape: (120, 128, 313)
number of label lists: 120
first segment labels: ['22961', '23158', '24321', '517063', '65380']


## Encoding the labels

A model cannot read species ID strings. The labels are converted to multi-hot vectors: a row with one position per species, set to 1 where that species is present. This is the natural encoding for a multi-label task, where a segment can contain several species at once.

In [11]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)   # learn the species set, then encode every segment

print("label matrix shape:", Y.shape)
print("distinct species across these files:", len(mlb.classes_))
print("first segment as multi-hot:", Y[0])
print("check, decoded back:", mlb.inverse_transform(Y[0:1]))

label matrix shape: (120, 21)
distinct species across these files: 21
first segment as multi-hot: [0 1 1 1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0]
check, decoded back: [('22961', '23158', '24321', '517063', '65380')]


## Saving the prepared data

The spectrogram stack, the label matrix and the species list are saved to disk, so later notebooks can load them directly rather than rebuilding from audio each time. These files sit in `data/`, which is gitignored, so they are not committed.

In [12]:
np.save('../data/toy_specs.npy', specs)
np.save('../data/toy_Y.npy', Y)

with open('../data/toy_classes.json', 'w') as f:
    json.dump(list(mlb.classes_), f)

print("saved specs, labels and class list to ../data")

saved specs, labels and class list to ../data
